# ALPHA-MATH — auditable Kaggle run

Run cells in order. This notebook performs preflight checks, regression tests, one-load real-model evaluation, optional competition inference, and finally creates a downloadable evidence bundle.

Required input: the ALPHA-MATH code ZIP and Qwen2.5-Math-7B-Instruct weights. Optional inputs: a labeled benchmark and AIMO `test.csv`.

In [ ]:
# EDIT ONLY THIS CELL when auto-discovery does not choose the right inputs.
MODEL_PATH = None       # folder containing config.json
BENCHMARK_PATH = None   # labeled .json/.jsonl/.csv; None = auto-discover, then bundled sanity set
TEST_CSV = None         # competition test.csv; None = auto-discover
EVAL_LIMIT = None       # use 3 for a quick real-model dry run
SUBMISSION_LIMIT = None # use 3 for a quick competition dry run
RUN_EVALUATION = True
RUN_SUBMISSION = True
RUN_ABLATION = False    # True is much stronger evidence but costs two extra evaluation passes

In [ ]:
import sys, zipfile
from pathlib import Path

WORK = Path('/kaggle/working')
repo_candidates = [p.parent.parent for p in Path('/kaggle/input').glob('**/src/agent.py')]
if repo_candidates:
    REPO = repo_candidates[0]
else:
    zips = list(Path('/kaggle/input').glob('**/AlphaMath_Kaggle_Bundle.zip'))
    if not zips:
        raise FileNotFoundError('Attach AlphaMath_Kaggle_Bundle.zip as a notebook input')
    with zipfile.ZipFile(zips[0]) as archive:
        archive.extractall(WORK)
    REPO = WORK / 'AlphaMath'
sys.path.insert(0, str(REPO))
print('REPO =', REPO)
assert (REPO / 'src' / 'agent.py').exists()

In [ ]:
# Resolve model weights without network access.
if MODEL_PATH is None:
    preferred = Path('/kaggle/input/models/urvishp80/qwen-2.5-math-7b/transformers/default/1')
    if preferred.exists():
        MODEL_PATH = str(preferred)
    else:
        configs = [p for p in Path('/kaggle/input').glob('**/config.json') if 'alphamath' not in str(p).lower()]
        ranked = [p.parent for p in configs if any(k in str(p).lower() for k in ('qwen', 'math', 'deepseek'))]
        MODEL_PATH = str(ranked[0]) if ranked else None
print('MODEL_PATH =', MODEL_PATH)
assert MODEL_PATH and (Path(MODEL_PATH) / 'config.json').exists(), 'Attach model weights or set MODEL_PATH'

In [ ]:
# Regression tests run before the expensive model load.
import os, subprocess
env = dict(os.environ, PYTHONPATH=str(REPO), PYTHONIOENCODING='utf-8')
completed = subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-v'], cwd=REPO, env=env)
assert completed.returncode == 0, 'Regression tests failed; do not trust this run'

In [ ]:
# Full experiment. The model is loaded once and reused for evaluation + submission.
from src.kaggle_experiment import run_kaggle_experiment
result = run_kaggle_experiment(
    REPO / 'configs' / 'kaggle.yaml',
    model_path=MODEL_PATH,
    benchmark_path=BENCHMARK_PATH,
    test_csv=TEST_CSV,
    output_dir='/kaggle/working/alphamath_artifacts',
    eval_limit=EVAL_LIMIT,
    submission_limit=SUBMISSION_LIMIT,
    run_evaluation=RUN_EVALUATION,
    run_competition_submission=RUN_SUBMISSION,
    run_ablation=RUN_ABLATION,
)

In [ ]:
# Inspect and download the complete evidence bundle.
from IPython.display import FileLink, Markdown, display
report_path = Path('/kaggle/working/alphamath_artifacts/FINAL_REPORT.md')
display(Markdown(report_path.read_text(encoding='utf-8')))
display(FileLink('/kaggle/working/alphamath_artifacts.zip'))
print('Download /kaggle/working/alphamath_artifacts.zip before closing the session.')